# Indexing

Heißt das so? Die Daten müssen jetzt in die Datenbank.

- DB: ChromaDB

Daten laden und für die DB aufbereiten. Das Schema sieht wie folgt aus:

```python
collection.add(
    documents=[...]
    metadatas=[...]
    ids=[...]
)
```

In [ ]:
import json
import random
import chromadb

from chromadb.config import Settings

In [ ]:
# ChromaDB initiallisieren
client = chromadb.PersistentClient(path="../data/vector_store")

# Collection erstellen
client.delete_collection(name="ProduktRAG")
collection = client.get_or_create_collection(
    name="ProduktRAG",
    metadata={"description": "Collection mit allen Beschreibungen und techn. Daten für das ProduktRAG"}
)

## Dataprep

In [ ]:
# Chunks laden
products_embedded = []

with open("../data/processed/products_embedded.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        products_embedded.append(json.loads(line))

print(f"{len(products_embedded)} Chunks geladen")

In [ ]:
# Chroma-Schema bauen
documents, embeddings, metadatas, ids = [], [], [], []

for i, chunk in enumerate(products_embedded):
    ids.append(chunk['id'])
    documents.append(chunk['document'])
    embeddings.append(chunk['embedding'])
    metadatas.append(chunk['metadata'])

## Daten in die DB bringen

In [ ]:
# DB max batch = 5000, daher in zwei iterationen
batch_total = len(products_embedded)
batch_size = 3000

for i in range(0, batch_total, batch_size):
    batch_end = min(i + batch_size, batch_total)

    # Daten speichern
    collection.add(
        ids=ids[i:batch_end],
        embeddings=embeddings[i:batch_end],
        metadatas=metadatas[i:batch_end],
        documents=documents[i:batch_end]
    )

print(f"{collection.count()} Chunks in der DB")

## Evaluation

In [ ]:
print(f"Anzahl der Chunks: {collection.count()}")
print()

# Chunks vergleichen
random_chunks = random.sample(products_embedded, k=5)

for i in range(len(random_chunks)):

    db_chunk = collection.get(ids=random_chunks[i]['id'])

    print(f"{random_chunks[i]['id']}")
    print(f"{random_chunks[i]['document']}")
    print(f"{db_chunk['documents'][0]}")
    print(f"{random_chunks[i]['metadata']}")
    print(f"{db_chunk['metadatas'][0]}")
    print()
    # Längenvergleich der Strings visuell